# IJCAI 2027 — Universal Kinematic TTC-Cone (Input) & TTC-OSC (Output) Benchmark

Notebook này chạy **song song với 2 kernel trên Kaggle**, tự động thực hiện trọn gói phần thực nghiệm trên **20 mô hình ngoại vi công bố (2020–2024)**:
1. **Output Interface (`TTC-OSC` + Định lý `JADE`) trên toàn bộ 20 mô hình × 5 tập ETH/UCY:** Kiểm chứng tính bất định mẫu biên–đồng thời (Marginal-to-Joint Unidentifiability) và ghép cặp mẫu tối ưu `TTC-OSC` không cần huấn luyện lại.
2. **Input Interface (`Raw` vs. `Static-3m` vs. `Kinematic TTC-Cone`) trên 11 mô hình đại diện đủ 4 cơ chế lỗi:** Chứng minh `TTC-Cone` giữ trọn phản xạ tránh va chạm tầm xa (các cặp lao nhanh vào nhau ở khoảng cách `> 3 m`) đồng thời triệt tiêu nhiễu người qua đường ngoài nón về đúng `0.000000 m` và bất biến thứ tự ID `0.000000 m`.

### Hướng dẫn chạy:
1. Vào **Runtime → Change runtime type** → chọn **GPU**.
2. Bấm **Runtime → Run all**.
3. Khi chạy xong Ô 4, tải file gọn nhẹ `universal_ijcai_summary.zip` trong `MyDrive/TTC_STGCNN_IJCAI2027/` về máy tính.

In [ ]:
# Ô 1: Mount Google Drive, cập nhật code mới nhất từ GitHub (idea1-colab) & chạy Self-Test
import os, sys, subprocess, shutil, glob, json
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/Thanh-2312/TTC-STGCNN.git'
BRANCH = 'idea1-colab'
DRIVE_BASE = '/content/drive/MyDrive/TTC_STGCNN_IJCAI2027'
REPO_DIR = os.path.join(DRIVE_BASE, 'repo')
WORK_DIR = os.path.join(DRIVE_BASE, 'univ_work')
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

RUNNER = os.path.join(REPO_DIR, 'colab', 'universal', 'run_univ.py')
ANALYZER = os.path.join(REPO_DIR, 'colab', 'universal', 'analyze_univ.py')
assert os.path.exists(RUNNER), f'Không tìm thấy {RUNNER}'

import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU', '| CPU cores:', os.cpu_count())
assert torch.cuda.is_available(), 'Chưa bật GPU! Hãy chọn Runtime -> Change runtime type -> GPU.'

subprocess.run([sys.executable, ANALYZER, '--selftest'], check=True)

def run_univ(**kw):
    env = dict(os.environ, **{k: str(v) for k, v in kw.items()})
    p = subprocess.Popen([sys.executable, '-u', RUNNER], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

In [ ]:
# Ô 2: Smoke Test nhanh (~3-5 phút trên tập eth) để kiểm tra toàn bộ 20 mô hình + Wrapper TTC-Cone + TTC-OSC
SMOKE_DIR = '/content/univ_smoke'
shutil.rmtree(SMOKE_DIR, ignore_errors=True)
shutil.rmtree('/content/univ_pert_smoke', ignore_errors=True)
workers = max(4, min(12, (os.cpu_count() or 4)))
rc = run_univ(UNIV_WORK=SMOKE_DIR, UNIV_PERT='/content/univ_pert_smoke', UNIV_DS='eth', UNIV_SMOKE='1', UNIV_WORKERS=workers)

failed_s = json.load(open(os.path.join(SMOKE_DIR, 'failed.json'))) if os.path.exists(os.path.join(SMOKE_DIR, 'failed.json')) else ['missing']
print('\n=== KẾT QUẢ SMOKE TEST ===')
print('Số job lỗi:', len(failed_s), failed_s)
for key in failed_s[:5]:
    lf = os.path.join(SMOKE_DIR, 'logs', f'{key}.txt')
    if os.path.exists(lf):
        print(f'\n--- Log lỗi {key} ---')
        print(''.join(open(lf).readlines()[-20:]))
if os.path.exists(os.path.join(SMOKE_DIR, 'results.md')):
    print('\n' + open(os.path.join(SMOKE_DIR, 'results.md'), encoding='utf-8').read())

In [ ]:
# Ô 3: Chạy ĐẦY ĐỦ trên cả 5 tập ETH/UCY — lưu trực tiếp vào Google Drive (hỗ trợ chạy tiếp nếu ngắt kết nối)
workers = max(4, min(14, (os.cpu_count() or 4)))
print(f'Bắt đầu chạy đầy đủ vào {WORK_DIR} với {workers} tiến trình song song...')
run_univ(UNIV_WORK=WORK_DIR, UNIV_PERT='/content/univ_pert_full', UNIV_WORKERS=workers)

res_md = os.path.join(WORK_DIR, 'results.md')
if os.path.exists(res_md):
    print('\n' + '='*80 + '\n')
    print(open(res_md, encoding='utf-8').read())
res_png = os.path.join(WORK_DIR, 'results_overview.png')
if os.path.exists(res_png):
    from IPython.display import Image, display
    display(Image(filename=res_png))

In [ ]:
# Ô 4: Đóng gói kết quả gọn nhẹ trên Google Drive để tải về máy tính
import zipfile

summary_zip = os.path.join(DRIVE_BASE, 'universal_ijcai_summary.zip')
with zipfile.ZipFile(summary_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in ('results.json', 'results.md', 'results_overview.pdf', 'results_overview.png', 'commits.json', 'failed.json'):
        fp = os.path.join(WORK_DIR, fn)
        if os.path.exists(fp):
            z.write(fp, fn)

done_json = len(glob.glob(os.path.join(WORK_DIR, 'out', '*', '*', '*.json')))
print(f'Đã hoàn thành {done_json} job files.')
print('File kết quả gọn nhẹ (tải file này về máy):', summary_zip, f'({os.path.getsize(summary_zip)/1024:.1f} KB)')